# Detecting Idioms in Sentence

To replace idioms in a sentence, we first have to ***detect*** and ***locate*** idioms.

There are two approaches to this:

1. Fuzzy matching

2. BERT (Bidirectional Encoder Representation from Transformers)

In [11]:
from rapidfuzz import process, fuzz
import spacy
import nltk
from lemminflect import getInflection
from idiom_parser import *

nlp = spacy.load("en_core_web_sm")

idiomParser = Idioms()

def getBestDefinition(sentence: str):
    return idiomParser.find_best_definition(sentence)

def replaceIdiom(sentence, idiom, definition):
    if (definition.lower().startswith("to")):
        definition = definition[3:].strip()

    definition = definition.replace(".", "")

    doc = nlp(sentence)
    idiom_doc = nlp(idiom)
    def_doc = nlp(definition)

    print(def_doc[0].pos_)

    target_verb = None
    for token in doc:
        if (token.lemma_ == idiom_doc[0].lemma_):
            target_verb = token
            break
    
    if target_verb:
        tag = target_verb.tag_

        def_verb = def_doc[0].lemma_

        inflected_verb = getInflection(def_verb, tag=tag)[0]

        new_phrase = inflected_verb + " " + " ".join([t.text for t in def_doc[1:]])

        return sentence.replace(idiom, new_phrase).strip()
    
    return sentence.replace(idiom, definition).strip()

sentence = "He's chasing the american dream."
matches = idiomParser.find_idiom_matches(sentence)

for match in matches:
    result = replaceIdiom(sentence, match)
    print(result)

[nltk_data] Downloading package brown to /Users/connor/nltk_data...
[nltk_data]   Package brown is already up-to-date!


DET
He's chasing the a philosophy that with hard work , courage and determination , anyone can prosper and achieve success.


In [ ]:
import requests

def llm_fix_grammar(sentence, idiom_matches):
    for match in idiom_matches:
        idiom = match["text"]
        definition = match["definition"]

        prompt = f"""
        Original sentence: "{sentence}"
        Replace the phrase "{idiom}" with the definition "{definition}".
        Rewrite the sentence so that it is grammatically perfect and natural.
        Only return the corrected sentence. No explanation.
        """

        res = requests.post("http://localhost:11434/api/generate",
                                 json={
                                     "model": "llama3",
                                     "prompt": prompt,
                                     "stream": False
                                 })
        if (res.status_code == 200):
            sentence = res.json()["response"].strip()
        
    return sentence

sentence = "He's chasing the american dream."
matches = idiomParser.find_idiom_matches(sentence)

res = llm_fix_grammar(sentence, matches)
print(res)



200
He's chasing the american dream.
